# MedRouteBench — Staged Evaluation (interactive demo)

This notebook is a **thin wrapper** around the `staged_eval` Python package.
All logic lives in the package — edit `config.py`, `prompts.py`, `runner.py`,
etc. to change behaviour.

**CLI alternative** (faster for larger runs):
```bash
cd MedRouteBench
python -m staged_eval.pipeline --n 50 --inspect 0
```

**Requires:** `GROQ_API_KEY` in `MedRouteBench/.env` (see `.env.example`).


In [ ]:
# One-time install (uncomment if needed).
# %pip install -q groq python-dotenv tenacity


In [3]:
import os, sys
from pathlib import Path

# Ensure the MedRouteBench root is on the path so the package resolves.
_root = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from staged_eval import run_pipeline, inspect_trace
from staged_eval.config import GROQ_API_KEY, GROQ_MODEL, TEST_SET_PATH

print(f"model : {GROQ_MODEL}")
print(f"data  : {TEST_SET_PATH}")
print(f"key   : {'set ✓' if GROQ_API_KEY else 'NOT SET ✗'}")


model : llama-3.1-8b-instant
data  : /Users/chuanhaixu/Documents/algoverse/MedRouteBench/pubmedqa/data/test_set.json
key   : set ✓


## Run pipeline

Adjust `N_RUN` and `STRATIFY` below. Artifacts are written to `staged_eval/runs/<UTC-ts>/`.


In [4]:
N_RUN    = 50
STRATIFY = True

report, traces = run_pipeline(n=N_RUN, stratify=STRATIFY, sleep_seconds=15.0)


run_dir: /Users/chuanhaixu/Documents/algoverse/MedRouteBench/staged_eval/runs/20260707T183137Z  |  n_cases: 50
eligible cases: 482  |  skipped ineligible: 18
stratified sample: {'yes': 28, 'no': 17, 'maybe': 5}
split strategies: {'results_boundary': 50}
sleep between cases: 15.0s
[1/50] pmid=12377809  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[2/50] pmid=26163474  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[3/50] pmid=19100463  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[4/50] pmid=18537964  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[5/50] pmid=12913878  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[6/50] pmid=25475395  gt=yes  split=results_boundary  stage1_chunks=3  stage2_chunks=1
[7/50] pmid=19130332  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[8/50] pmid=9427037  gt=yes  split=results_boundary  stage1_chunks=2  stage2_chunks=1
[9/50] pmid=24481006  gt

## Inspect a trace (optional)


In [33]:
# Inspect the first trace (change index as desired).

index = 16

inspect_trace(traces[index]) # summary
traces[index] # full trace


pmid=18041059  gold=yes  completed=True  label=overreaction
  stage1: action=ANSWER         answer=yes    conf=0.8 
  stage2: action=REVISE_ANSWER  answer=no     conf=0.9 


{'pmid': '18041059',
 'gold_label': 'yes',
 'question': 'Do adjuvant aromatase inhibitors increase the cardiovascular risk in postmenopausal women with early breast cancer?',
 'split_strategy': 'results_boundary',
 'completed': True,
 'stage1_evidence_shown': [{'label': 'Background',
   'context': 'Despite the advantages from using aromatase inhibitors (AIs) compared with tamoxifen for early breast cancer, an unexpectedly greater number of grade 3 and 4 cardiovascular events (CVAE) (as defined by National Cancer Institute of Canada-Common Toxicity Criteria [version 2.0] was demonstrated.'},
  {'label': 'Methods',
   'context': 'Phase 3 randomized clinical trials (RCTs) comparing AI with tamoxifen in early breast cancer were considered eligible for this review. The event-based risk ratios (RRs) with 95% confidence intervals (95% CIs) were derived, and a test of heterogeneity was applied. Finally, absolute differences (ADs) in event rates and the number of patients needed to harm 1 patie